# Song Recommendation System — Exploration

This notebook is for **exploration and visualization only**. All core logic lives in `src/` as importable, tested modules — see `scripts/train.py` for the end-to-end pipeline this notebook walks through interactively.

In [ ]:
import sys
sys.path.append('..')  # so `src` is importable when running from notebooks/

import numpy as np
import matplotlib.pyplot as plt

from src.data_loader import SongRecommendationDataGenerator
from src.dimensionality import DimensionalityReducer
from src.kernels import MultipleKernelLearning
from src.theory import VCDimensionAnalyzer
from src.model import ConvexRecommendationModel
from src.baselines import BaselineModels
from src.recommender import SongRecommender

np.random.seed(42)


## 1. Generate synthetic data

In [ ]:
data_gen = SongRecommendationDataGenerator(n_users=1000, n_songs=5000)
data_gen.generate_data()


## 2. Dimensionality reduction

In [ ]:
song_reducer = DimensionalityReducer(n_components=20)
song_features_reduced = song_reducer.fit_transform(data_gen.song_features, method='sparse_pca')

user_reducer = DimensionalityReducer(n_components=10)
user_features_reduced = user_reducer.fit_transform(data_gen.user_features, method='pca')

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].scatter(song_features_reduced[:, 0], song_features_reduced[:, 1], alpha=0.5, s=10, c='blue')
axes[0].set_title('Song Features (Sparse PCA)')
axes[1].scatter(user_features_reduced[:, 0], user_features_reduced[:, 1], alpha=0.5, s=10, c='red')
axes[1].set_title('User Features (PCA)')
plt.tight_layout()
plt.show()


## 3. Multiple Kernel Learning

In [ ]:
mkl = MultipleKernelLearning(kernel_types=['rbf', 'polynomial', 'linear'])
subset_size = min(500, len(song_features_reduced))
X_subset = song_features_reduced[:subset_size]
combined_kernel = mkl.compute_combined_kernel(X_subset)

fig, axes = plt.subplots(1, 4, figsize=(20, 5))
kernels = mkl.compute_kernels(X_subset)
names = ['RBF', 'Polynomial', 'Linear', 'Combined']
for ax, K, name in zip(axes, kernels + [combined_kernel], names):
    im = ax.imshow(K[:50, :50], cmap='hot', aspect='auto')
    ax.set_title(name)
    plt.colorbar(im, ax=ax)
plt.tight_layout()
plt.show()


## 4. VC-dimension analysis

In [ ]:
vc_analyzer = VCDimensionAnalyzer()
vc_results = vc_analyzer.analyze_model(
    rank=10, n_nonzero=20, n_features=50, n_samples=len(data_gen.train_triples)
)
for key, value in vc_results.items():
    print(f"{key}: {value:.2f}")


## 5. Train the core model

In [ ]:
model = ConvexRecommendationModel(
    n_users=1000, n_songs=5000,
    n_features_song=data_gen.n_features_song,
    lambda_nuc=1.0, lambda_l1=0.1,
)
model.fit_accelerated_proximal_sgd(
    triples=data_gen.train_triples[:5000],
    features=data_gen.song_features,
    n_epochs=50, batch_size=200, learning_rate=0.001,
)

plt.figure(figsize=(10, 5))
plt.plot(model.losses, 'b-', linewidth=2)
plt.xlabel('Epoch'); plt.ylabel('BPR Loss'); plt.title('Training Loss Convergence')
plt.grid(True, alpha=0.3)
plt.show()


## 6. Evaluate and compare to baselines

In [ ]:
test_subset = data_gen.test_triples[:1000]
metrics = model.evaluate(test_subset, data_gen.song_features, k=10)
print(f"Hit Rate@10: {metrics['hit_rate']:.3f} | NDCG@10: {metrics['ndcg']:.3f}")

popular_songs = BaselineModels.popularity_based(data_gen.interactions, k=10)
pop_hit = np.mean([1 if t[1] in popular_songs else 0 for t in test_subset])

random_hits = [
    np.mean([1 if t[1] in BaselineModels.random_recommendation(data_gen.n_songs, k=10) else 0 for t in test_subset])
    for _ in range(10)
]

import pandas as pd
df_results = pd.DataFrame({
    'Method': ['Proposed (SRM+MKL)', 'Popularity', 'Random'],
    'Hit Rate': [metrics['hit_rate'], pop_hit, np.mean(random_hits)],
})
df_results.plot(x='Method', y='Hit Rate', kind='bar', legend=False, title='Hit Rate@10 Comparison')
plt.tight_layout(); plt.show()
print(df_results)


## 7. Structural Risk Minimization sweep

In [ ]:
lambda_nuc_values = [0.01, 0.1, 1.0, 10.0]
lambda_l1_values = [0.001, 0.01, 0.1, 1.0]

train_errors, test_errors, complexities = [], [], []
for l_nuc in lambda_nuc_values:
    for l_l1 in lambda_l1_values:
        m = ConvexRecommendationModel(
            n_users=1000, n_songs=5000, n_features_song=data_gen.n_features_song,
            lambda_nuc=l_nuc, lambda_l1=l_l1,
        )
        m.fit_accelerated_proximal_sgd(
            triples=data_gen.train_triples[:1000], features=data_gen.song_features,
            n_epochs=1, batch_size=100, learning_rate=0.001,
        )
        nuc_norm = np.linalg.norm(m.X, 'nuc')
        sparsity = np.sum(np.abs(m.w) > 1e-5)
        complexities.append(nuc_norm + sparsity)
        train_errors.append(m.losses[-1])
        test_errors.append(1 - m.evaluate(test_subset[:100], data_gen.song_features)['hit_rate'])

idx = np.argsort(complexities)
plt.figure(figsize=(10, 6))
plt.plot(np.array(complexities)[idx], np.array(train_errors)[idx], 'b.-', label='Training Error')
plt.plot(np.array(complexities)[idx], np.array(test_errors)[idx], 'r.-', label='Test Error')
plt.xlabel('Model Complexity (Nuclear Norm + Sparsity)'); plt.ylabel('Error')
plt.title('Structural Risk Minimization'); plt.legend(); plt.grid(alpha=0.3)
plt.show()


## 8. Sample recommendations

In [ ]:
recommender = SongRecommender(model, data_gen.song_features, song_urls=data_gen.song_urls)
for user_id in [0, 100, 500]:
    print(f"Recommendations for User {user_id}:")
    recs = recommender.recommend_for_user(user_id, n_recommendations=5)
    print(recs[['song', 'score', 'url']].to_string(index=False))
    print()
